# BERTurk ile Türkçe Yorum Puan Sınıflandırıcı (1–5★)

Faz 2: Transformer fine-tune. **Colab'da GPU açın** (Runtime > Change runtime type > GPU).

Klasik baseline ~%64.5 exact / %90 ±1 verdi. BERTurk bağlamı ve olumsuzluğu
doğal yakaladığı için exact accuracy'de sıçrama bekliyoruz.


## 1. Kurulum & GPU kontrol

In [1]:
!pip -q install transformers datasets accelerate evaluate
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "YOK — GPU açın!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.2 MB/s eta 0:00:00
GPU: NVIDIA L4


## 2. Veri yükle (Drive)

In [2]:
from google.colab import drive
drive.mount('/content/drive')

CSV = '/content/drive/MyDrive/turkish_ecommerce_reviews/turkish_ecommerce_reviews.csv'

import pandas as pd
df = pd.read_csv(CSV).rename(columns={'Rating (Star)': 'rating'})
df = df.dropna(subset=['Review', 'rating'])
print(len(df), 'satır')

Mounted at /content/drive
272216 satır


## 3. Veri hijyeni

Transformer için **ağır temizlik YAPMIYORUZ** — BERTurk cased modeli büyük/küçük
harf ve noktalamadan faydalanır. Sadece dedup + çok kısa yorumları atıyoruz.
Olumsuzluk kelimeleri elbette korunur.

In [3]:
df['text'] = df['Review'].astype(str).str.strip()
df = df[df['text'].str.len() >= 2]
before = len(df)
df = df.drop_duplicates(subset=['text']).reset_index(drop=True)
print(f'Dedup: {before:,} -> {len(df):,}')

# Etiket 0..4 (model 0-index ister), gerçek yıldız = label+1
df['label'] = df['rating'].astype(int) - 1
print(df['rating'].value_counts(normalize=True).sort_index().round(3))

Dedup: 272,216 -> 238,520
rating
1    0.026
2    0.025
3    0.108
4    0.235
5    0.608
Name: proportion, dtype: float64


## 4. Train / Val / Test split (stratified)

In [4]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])
print('train', len(train_df), 'val', len(val_df), 'test', len(test_df))

train 190816 val 23852 test 23852


## 5. Tokenizer & Dataset

In [5]:
from transformers import AutoTokenizer
from datasets import Dataset

MODEL_NAME = 'dbmdz/bert-base-turkish-uncased'   # veri zaten küçük harfli -> eşleşir
MAX_LEN = 160   # 128 bazı uzun yorumları kesiyordu; 160 daha iyi kapsar

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_ds(d):
    return Dataset.from_pandas(d[['text', 'label']].reset_index(drop=True))

def tok(batch):
    return tokenizer(batch['text'], truncation=True, max_length=MAX_LEN)

ds_train = to_ds(train_df).map(tok, batched=True)
ds_val   = to_ds(val_df).map(tok, batched=True)
ds_test  = to_ds(test_df).map(tok, batched=True)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/263k [00:00<?, ?B/s]

Map:   0%|          | 0/190816 [00:00<?, ? examples/s]

Map:   0%|          | 0/23852 [00:00<?, ? examples/s]

Map:   0%|          | 0/23852 [00:00<?, ? examples/s]

## 6. Model + sınıf ağırlıkları

Aşırı dengesiz veri (5★ %60, 2★ %2.5). Azınlık sınıflarını (olumsuz yorumlar =
sahte-puan tespiti için kritik) kaçırmamak için **ağırlıklı CrossEntropy** kullanan
özel bir Trainer tanımlıyoruz.

In [6]:
import numpy as np, torch
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.utils.class_weight import compute_class_weight

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=5)

classes = np.array([0, 1, 2, 3, 4])
cw = compute_class_weight('balanced', classes=classes, y=train_df['label'].values)
cw = np.sqrt(cw)                                  # YUMUŞAT: accuracy korunur, recall iyi kalır
class_weights = torch.tensor(cw, dtype=torch.float)
print('softened class weights:', cw.round(2))

LABEL_SMOOTHING = 0.1   # komşu-yıldız etiket gürültüsüne (4★ vs 5★) karşı

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        loss = torch.nn.functional.cross_entropy(
            outputs.logits, labels,
            weight=class_weights.to(outputs.logits.device),
            label_smoothing=LABEL_SMOOTHING)
        return (loss, outputs) if return_outputs else loss


pytorch_model.bin:   0%|          | 0.00/445M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: dbmdz/bert-base-turkish-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were new

softened class weights: [2.8  2.86 1.36 0.92 0.57]


## 7. Metrikler

In [7]:
import numpy as np
from sklearn.metrics import f1_score, mean_absolute_error

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    preds = probs.argmax(-1)
    ev = probs @ np.array([0, 1, 2, 3, 4])        # beklenen değer (0-indeksli)
    return {
        'accuracy': (preds == labels).mean(),
        'off_by_one': (np.abs(preds - labels) <= 1).mean(),
        'mae': mean_absolute_error(labels, preds),
        'ev_mae': mean_absolute_error(labels, ev),
        'macro_f1': f1_score(labels, preds, average='macro'),
    }


## 8. Eğitim

GPU'da 2 epoch ~20–40 dk sürebilir (veri boyutuna göre). `fp16=True` hızlandırır.

In [8]:
from transformers import DataCollatorWithPadding
collator = DataCollatorWithPadding(tokenizer)

args = TrainingArguments(
    output_dir='/content/berturk_out',
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    fp16=True,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    logging_steps=200,
    report_to='none',
)

trainer = WeightedTrainer(
    model=model, args=args,
    train_dataset=ds_train, eval_dataset=ds_val,
    processing_class=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
)
trainer.train()


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


model.safetensors:   0%|          | 0.00/445M [00:00<?, ?B/s]

Epoch,Training Loss,Validation Loss,Accuracy,Off By One,Mae,Ev Mae,Macro F1
1,1.420480,1.425304,0.668707,0.927427,0.413928,0.743513,0.487045
2,1.387919,1.413309,0.668833,0.933632,0.408938,0.783513,0.497911
3,1.327223,1.433774,0.667449,0.937951,0.402817,0.737320,0.498932
4,1.251092,1.479762,0.637934,0.940718,0.429398,0.771324,0.504557
5,1.199919,1.512605,0.648583,0.935645,0.423235,0.745050,0.505336


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=29815, training_loss=1.3327733211462958, metrics={'train_runtime': 3513.2638, 'train_samples_per_second': 271.565, 'train_steps_per_second': 8.486, 'total_flos': 6.647755515818918e+16, 'train_loss': 1.3327733211462958, 'epoch': 5.0})

## 9. Test seti değerlendirmesi

In [9]:
import pandas as pd
print(trainer.evaluate(ds_test))

from sklearn.metrics import classification_report, confusion_matrix
pred = trainer.predict(ds_test)
y_pred = pred.predictions.argmax(-1)
y_true = pred.label_ids
print(classification_report(y_true, y_pred, target_names=['1*','2*','3*','4*','5*']))
print(confusion_matrix(y_true, y_pred))

{'eval_loss': 1.518510341644287, 'eval_accuracy': 0.6453546872379675, 'eval_off_by_one': 0.9353513332215327, 'eval_mae': 0.4265051148750629, 'eval_ev_mae': 0.7453664727400181, 'eval_macro_f1': 0.4976777681803416, 'eval_runtime': 27.5369, 'eval_samples_per_second': 866.184, 'eval_steps_per_second': 13.545, 'epoch': 5.0}
              precision    recall  f1-score   support

          1*       0.63      0.56      0.60       611
          2*       0.31      0.35      0.33       584
          3*       0.37      0.29      0.32      2567
          4*       0.43      0.47      0.45      5595
          5*       0.79      0.79      0.79     14495

    accuracy                           0.65     23852
   macro avg       0.51      0.49      0.50     23852
weighted avg       0.64      0.65      0.64     23852

[[  344   144    78    22    23]
 [   75   205   217    69    18]
 [   86   232   741   772   736]
 [   13    58   601  2628  2295]
 [   27    19   393  2581 11475]]


## 10. Modeli kaydet (Drive)

İndirip lokal FastAPI'de `inference_berturk.py` ile servis edebilirsin.

In [10]:
SAVE = '/content/drive/MyDrive/turkish_ecommerce_reviews/berturk_model'
trainer.save_model(SAVE)
tokenizer.save_pretrained(SAVE)
print('Kaydedildi:', SAVE)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Kaydedildi: /content/drive/MyDrive/turkish_ecommerce_reviews/berturk_model
